# Children's Phoneme ASR — Full Pipeline
CNN + BiLSTM + CTC Loss

**Sections:** Install → Data → Features → Model → Train → Inference

## 1. Install Dependencies

In [6]:
!pip install librosa torch torchaudio -q

In [7]:
import json

def split_phones(text):
    phones = []
    i = 0
    while i < len(text):
        if i + 1 < len(text) and text[i+1] == "ː":  # handle long vowels
            phones.append(text[i:i+2])
            i += 2
        elif text[i] != " ":
            phones.append(text[i])
            i += 1
        else:
            i += 1
    return " ".join(phones)


def convert_jsonl_to_mapping(input_path):
    mapping = {}

    with open(input_path, "r") as f:
        for line in f:
            record = json.loads(line)

            filename = record["utterance_id"] + ".flac"  # match expected format
            phonemes = split_phones(record["phonetic_text"])

            mapping[filename] = phonemes

    return mapping

## 2. Imports & Config

In [8]:
import os
import json
import random
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from pathlib import Path


# ── Config ──────────────────────────────────────────────
AUDIO_DIR   = Path('audio/')
LABELS_JSON = Path('train_phon_transcripts.jsonl')
SAMPLE_RATE = 16000
N_MFCC      = 40
BATCH_SIZE  = 16
EPOCHS      = 30
LR          = 1e-3
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(LABELS_JSON)

Using device: cuda
train_phon_transcripts.jsonl


## 3. Build Vocabulary from Labels

Reads all IPA tokens in your JSON, assigns each an integer index.
`0` is always reserved for the CTC blank token.

In [9]:
labels_dict = convert_jsonl_to_mapping(LABELS_JSON)
# labels_dict expected format:
# { 'file1.wav': 'h ɛ l oʊ', 'file2.wav': 'w ɜ r l d', ... }
# i.e. IPA phones are space-separated strings

# Collect all unique phones
all_phones = set()
for phone_str in labels_dict.values():
    for phone in phone_str.strip().split():
        all_phones.add(phone)

# 0 = CTC blank, phones start at 1
vocab       = ['<blank>'] + sorted(all_phones)
phone2idx   = {p: i for i, p in enumerate(vocab)}
idx2phone   = {i: p for p, i in phone2idx.items()}
NUM_CLASSES = len(vocab)

print(f'Vocab size (including blank): {NUM_CLASSES}')
print(f'Sample phones: {vocab[1:10]}')

Vocab size (including blank): 83
Sample phones: ['b', 'bː', 'c', 'd', 'dː', 'e', 'eː', 'f', 'fː']


In [10]:
def safe_delta(mfcc):
    width = min(9, mfcc.shape[1])
    if width % 2 == 0:
        width -= 1
    if width < 3:
        width = 3
    return librosa.feature.delta(mfcc, width=width)

## 4. Dataset Class

Loads audio → extracts MFCC+delta+delta2 → encodes label to int tensor.

In [11]:
class PhonemeDataset(Dataset):
    def __init__(self, labels_dict, audio_dir, phone2idx,
                 sr=16000, n_mfcc=40):
        self.items     = list(labels_dict.items())  # [(filename, phone_str), ...]
        self.audio_dir = audio_dir
        self.phone2idx = phone2idx
        self.sr        = sr
        self.n_mfcc    = n_mfcc

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        filename, phone_str = self.items[idx]
        path = os.path.join(self.audio_dir, filename)

        # ── Load audio ──────────────────────────────────
        y, _ = librosa.load(path, sr=self.sr)

        # ── MFCC + delta + delta-delta ───────────────────
        mfcc   = librosa.feature.mfcc(y=y, sr=self.sr, n_mfcc=self.n_mfcc)
        delta = safe_delta(mfcc)
        delta2 = safe_delta(mfcc)
        feat   = np.vstack([mfcc, delta, delta2])  # (n_mfcc*3, time)
        feat   = feat.T                             # (time, n_mfcc*3)

        # Per-sample normalization (important for children's variable volumes)
        feat = (feat - feat.mean(axis=0)) / (feat.std(axis=0) + 1e-8)

        feat_tensor = torch.FloatTensor(feat)       # (time, 120)

        # ── Encode label ────────────────────────────────
        phones      = phone_str.strip().split()
        label_tensor = torch.LongTensor(
            [self.phone2idx[p] for p in phones if p in self.phone2idx]
        )

        return feat_tensor, label_tensor

print('Dataset class defined.')

Dataset class defined.


## 5. Collate Function + DataLoader

Audio clips have variable lengths — we pad them to the longest in the batch.
We also track the real lengths so CTC loss isn't confused by padding.

In [12]:
def collate_fn(batch):
    feats, labels = zip(*batch)

    # Lengths before padding
    feat_lengths  = torch.LongTensor([f.shape[0] for f in feats])
    label_lengths = torch.LongTensor([l.shape[0] for l in labels])

    # Pad features to longest in batch: (batch, max_time, 120)
    feats_padded  = pad_sequence(feats,  batch_first=True, padding_value=0.0)
    # Concatenate labels (CTC expects a flat 1D tensor)
    labels_concat = torch.cat(labels)

    return feats_padded, labels_concat, feat_lengths, label_lengths


# ── Train / Val split ──────────────────────────────────
all_items  = list(labels_dict.items())
random.shuffle(all_items)
split      = int(0.9 * len(all_items))
train_dict = dict(all_items[:split])
val_dict   = dict(all_items[split:])

train_ds = PhonemeDataset(train_dict, AUDIO_DIR, phone2idx)
val_ds   = PhonemeDataset(val_dict,   AUDIO_DIR, phone2idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

Train: 10838 samples | Val: 1205 samples


## 6. Model — CNN + BiLSTM

In [13]:
class CNNBiLSTMPhoneme(nn.Module):
    def __init__(self, n_features=120, num_classes=41,
                 lstm_hidden=256, lstm_layers=2):
        super().__init__()

        # ── CNN frontend ─────────────────────────────────
        # Input: (batch, 1, n_features, time)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,1)),   # halve freq axis, keep time

            nn.Conv2d(32, 64, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,1)),   # halve again

            nn.Dropout2d(0.25),
        )
        # After 2x pool on freq: n_features -> n_features//4
        cnn_out_dim = 64 * (n_features // 4)

        # ── BiLSTM ───────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=cnn_out_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        # x: (batch, time, n_features)
        x = x.unsqueeze(1)             # (batch, 1, time, n_features)
        x = x.permute(0, 1, 3, 2)     # (batch, 1, n_features, time)

        x = self.cnn(x)                # (batch, 64, n_features//4, time)

        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)     # (batch, time, 64, n_features//4)
        x = x.reshape(b, t, c * f)    # (batch, time, cnn_out_dim)

        x, _ = self.bilstm(x)         # (batch, time, lstm_hidden*2)
        x = self.fc(x)                 # (batch, time, num_classes)
        return x


N_FEATURES  = N_MFCC * 3   # mfcc + delta + delta2 = 120
model = CNNBiLSTMPhoneme(
    n_features=N_FEATURES,
    num_classes=NUM_CLASSES
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')

Model parameters: 6,099,091


## 7. Training Loop

In [14]:
import os

ctc_loss  = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)

best_val_loss = float('inf')
start_epoch = 1

# 🔁 Resume if checkpoint exists
if os.path.exists('last_checkpoint.pt'):
    checkpoint = torch.load('last_checkpoint.pt', map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    best_val_loss = checkpoint['best_val_loss']
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {start_epoch}")

for epoch in range(start_epoch, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for batch_idx, (feats, labels, feat_lens, label_lens) in enumerate(train_loader):
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        logits   = model(feats)
        log_prob = logits.log_softmax(-1).permute(1, 0, 2)

        loss = ctc_loss(log_prob, labels, feat_lens, label_lens)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_loss += loss.item()

        # 🔹 Batch logging (prevents idle timeout)
        if batch_idx % 20 == 0:
            print(f"[Epoch {epoch} | Batch {batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")

    # ── Validate ───────────────────────────────────────
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for feats, labels, feat_lens, label_lens in val_loader:
            feats  = feats.to(DEVICE)
            labels = labels.to(DEVICE)

            logits   = model(feats)
            log_prob = logits.log_softmax(-1).permute(1, 0, 2)

            loss = ctc_loss(log_prob, labels, feat_lens, label_lens)
            val_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    print(f"\nEpoch {epoch:03d}")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")
    print(f"  LR         : {current_lr:.6f}")

    # 💾 Save last checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss
    }, 'last_checkpoint.pt')

    # ⭐ Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  ✓ Saved best model (val_loss={val_loss:.4f})")

/home/rgcodes/anaconda3/envs/mlbasic/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Epoch 1 | Batch 0/678] Loss: 31.3224
[Epoch 1 | Batch 20/678] Loss: 4.1888
[Epoch 1 | Batch 40/678] Loss: 3.9445
[Epoch 1 | Batch 60/678] Loss: 3.8007
[Epoch 1 | Batch 80/678] Loss: 3.9644
[Epoch 1 | Batch 100/678] Loss: 3.9269
[Epoch 1 | Batch 120/678] Loss: 4.1208
[Epoch 1 | Batch 140/678] Loss: 4.0048
[Epoch 1 | Batch 160/678] Loss: 3.9413
[Epoch 1 | Batch 180/678] Loss: 4.2673
[Epoch 1 | Batch 200/678] Loss: 3.9449
[Epoch 1 | Batch 220/678] Loss: 4.0649
[Epoch 1 | Batch 240/678] Loss: 4.1750
[Epoch 1 | Batch 260/678] Loss: 3.7935
[Epoch 1 | Batch 280/678] Loss: 4.0808
[Epoch 1 | Batch 300/678] Loss: 3.8289
[Epoch 1 | Batch 320/678] Loss: 4.5819
[Epoch 1 | Batch 340/678] Loss: 3.8293
[Epoch 1 | Batch 360/678] Loss: 3.8619
[Epoch 1 | Batch 380/678] Loss: 4.0339
[Epoch 1 | Batch 400/678] Loss: 3.9655
[Epoch 1 | Batch 420/678] Loss: 3.9468
[Epoch 1 | Batch 440/678] Loss: 3.8347
[Epoch 1 | Batch 460/678] Loss: 3.7873
[Epoch 1 | Batch 480/678] Loss: 4.2508
[Epoch 1 | Batch 500/678] Loss

## 8. Inference — Greedy CTC Decode

Greedy decode: take the argmax at each timestep, collapse repeats, remove blanks.

In [16]:
def greedy_ctc_decode(logits, idx2phone, blank_idx=0):
    """
    logits: (time, num_classes) — single sample, already on CPU
    Returns: list of predicted phone strings
    """
    indices = logits.argmax(-1).tolist()   # argmax at each timestep

    # Collapse consecutive duplicates, then remove blanks
    phones = []
    prev = None
    for idx in indices:
        if idx != prev:
            if idx != blank_idx:
                phones.append(idx2phone[idx])
            prev = idx

    return phones


def predict(audio_path, model, phone2idx, idx2phone,
            sr=16000, n_mfcc=40, device='cpu'):
    model.eval()

    # Extract features (same as training)
    y, _ = librosa.load(audio_path, sr=sr)
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2]).T            # (time, 120)
    feat   = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)  # normalize

    feat_tensor = torch.FloatTensor(feat).unsqueeze(0).to(device)  # (1, time, 120)

    with torch.no_grad():
        logits = model(feat_tensor)                 # (1, time, num_classes)
        logits = logits.squeeze(0).cpu()            # (time, num_classes)

    return greedy_ctc_decode(logits, idx2phone)


# ── Example usage ──────────────────────────────────────
# Load best saved model
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

test_file = 'audio/U_0a5f2a05463dac28.flac'   # replace with a real file
predicted_phones = predict(test_file, model, phone2idx, idx2phone, device=DEVICE)
print('Predicted phones:', ' '.join(predicted_phones))

Predicted phones: k ɪ ɑ i o i s ɑ i k ɪ t u s


## 9. CER Evaluation on Val Set

In [18]:
def compute_cer(predicted, reference):
    """
    predicted, reference: lists of phone strings
    Returns CER = (S + D + I) / N  (edit distance / ref length)
    """
    p, r = predicted, reference
    dp = [[0] * (len(r) + 1) for _ in range(len(p) + 1)]
    for i in range(len(p) + 1):
        dp[i][0] = i
    for j in range(len(r) + 1):
        dp[0][j] = j
    for i in range(1, len(p) + 1):
        for j in range(1, len(r) + 1):
            cost = 0 if p[i - 1] == r[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[len(p)][len(r)] / max(len(r), 1)


def _safe_delta_for_infer(mfcc, order=1):
    t = mfcc.shape[1]
    width = min(9, t if t % 2 == 1 else t - 1)
    if width < 3:
        return np.zeros_like(mfcc)
    return librosa.feature.delta(mfcc, width=width, order=order, mode='nearest')


def predict(audio_path, model, phone2idx, idx2phone, sr=16000, n_mfcc=40, device='cpu'):
    model.eval()

    y, _ = librosa.load(audio_path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = _safe_delta_for_infer(mfcc, order=1)
    delta2 = _safe_delta_for_infer(mfcc, order=2)

    feat = np.vstack([mfcc, delta, delta2]).T
    feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
    feat_tensor = torch.FloatTensor(feat).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(feat_tensor).squeeze(0).cpu()

    return greedy_ctc_decode(logits, idx2phone)


model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()

total_cer, n = 0.0, 0
for filename, phone_str in val_dict.items():
    path = os.path.join(AUDIO_DIR, filename)
    if not os.path.exists(path):
        continue
    reference = phone_str.strip().split()
    predicted = predict(path, model, phone2idx, idx2phone, device=DEVICE)
    total_cer += compute_cer(predicted, reference)
    n += 1

if n == 0:
    print('Val CER: N/A (no valid samples found)')
else:
    print(f'Val CER: {total_cer / n * 100:.2f}%  (over {n} samples)')

Val CER: 77.97%  (over 1205 samples)
